# Test Healthcare `/run` Endpoint

Use this notebook after starting the API server from the repository root:

```bash
python run.py
```

The notebook loops through `input_examples/input_example_*.json`, sends each payload to `POST http://localhost:8000/run`, and saves the final returned JSON under `output_examples/`.

In [25]:
from pathlib import Path
import json
import requests

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

BASE_URL = "http://localhost:8000"
RUN_ENDPOINT = f"{BASE_URL}/run"

print("Project root:", PROJECT_ROOT)
print("Run endpoint:", RUN_ENDPOINT)

Project root: /Users/vishvajit.phalke/phantom/source_codes/legal_intelligence/git_hub/legal_intelligence
Run endpoint: http://localhost:8000/run


## 1. Check API Health

Run this cell after `python run.py` is already running in a terminal.

In [26]:
health_response = requests.get(f"{BASE_URL}/health", timeout=10)
print("Status code:", health_response.status_code)
print(health_response.json())

Status code: 200
{'status': 'healthy'}


## 2. Discover Input Examples

This finds every `input_examples/input_example_*.json` file and prepares matching `output_examples/output_example_*.json` paths.

In [27]:
import re

INPUT_DIR = PROJECT_ROOT / "input_examples"
OUTPUT_DIR = PROJECT_ROOT / "output_examples"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def example_number(path: Path) -> int:
    match = re.search(r"input_example_(\d+)\.json$", path.name)
    return int(match.group(1)) if match else 10**9

input_paths = sorted(INPUT_DIR.glob("input_example_*.json"), key=example_number)
if not input_paths:
    raise FileNotFoundError(f"No input examples found in {INPUT_DIR}")

print(f"Found {len(input_paths)} input example(s):")
for input_path in input_paths:
    output_path = OUTPUT_DIR / f"output_example_{example_number(input_path)}.json"
    print(f"- {input_path.name} -> {output_path.name}")

Found 10 input example(s):
- input_example_1.json -> output_example_1.json
- input_example_2.json -> output_example_2.json
- input_example_3.json -> output_example_3.json
- input_example_4.json -> output_example_4.json
- input_example_5.json -> output_example_5.json
- input_example_6.json -> output_example_6.json
- input_example_7.json -> output_example_7.json
- input_example_8.json -> output_example_8.json
- input_example_9.json -> output_example_9.json
- input_example_10.json -> output_example_10.json


## 3. Run Examples One By One

Each input is sent sequentially. The notebook first tries the Agentathon-required JSON request body. If the current local app still expects FastAPI form fields, it retries with form data so testing continues with the existing implementation.

In [28]:
def call_run_endpoint(payload: dict) -> tuple[dict, str, int]:
    response = requests.post(RUN_ENDPOINT, json=payload, timeout=900)
    request_mode = "json"

    if response.status_code == 422:
        response = requests.post(RUN_ENDPOINT, data=payload, timeout=900)
        request_mode = "form-fallback"

    try:
        response_json = response.json()
    except ValueError:
        print(response.text)
        raise

    if response.status_code >= 400:
        print(json.dumps(response_json, indent=2, ensure_ascii=False))
        response.raise_for_status()

    return response_json, request_mode, response.status_code

generated_outputs = []

for input_path in input_paths:
    number = example_number(input_path)
    output_path = OUTPUT_DIR / f"output_example_{number}.json"

    with input_path.open("r", encoding="utf-8") as file:
        payload = json.load(file)

    print(f"\nRunning {input_path.name}...")
    response_json, request_mode, status_code = call_run_endpoint(payload)

    with output_path.open("w", encoding="utf-8") as file:
        json.dump(response_json, file, indent=4, ensure_ascii=False)

    generated_outputs.append({
        "input": input_path.name,
        "output": output_path.name,
        "request_mode": request_mode,
        "status_code": status_code,
        "run_id": response_json.get("run_id"),
        "status": response_json.get("status"),
    })

    print(f"Saved {output_path}")

print("\nCompleted all examples.")


Running input_example_1.json...
Saved /Users/vishvajit.phalke/phantom/source_codes/legal_intelligence/git_hub/legal_intelligence/output_examples/output_example_1.json

Running input_example_2.json...
Saved /Users/vishvajit.phalke/phantom/source_codes/legal_intelligence/git_hub/legal_intelligence/output_examples/output_example_2.json

Running input_example_3.json...
Saved /Users/vishvajit.phalke/phantom/source_codes/legal_intelligence/git_hub/legal_intelligence/output_examples/output_example_3.json

Running input_example_4.json...
Saved /Users/vishvajit.phalke/phantom/source_codes/legal_intelligence/git_hub/legal_intelligence/output_examples/output_example_4.json

Running input_example_5.json...
Saved /Users/vishvajit.phalke/phantom/source_codes/legal_intelligence/git_hub/legal_intelligence/output_examples/output_example_5.json

Running input_example_6.json...
Saved /Users/vishvajit.phalke/phantom/source_codes/legal_intelligence/git_hub/legal_intelligence/output_examples/output_example

## 4. Review Generated Outputs

This prints a compact summary of the generated output files and shows the last returned JSON for quick inspection.

In [7]:
print(json.dumps(generated_outputs, indent=2, ensure_ascii=False))

if generated_outputs:
    last_output_path = OUTPUT_DIR / generated_outputs[-1]["output"]
    with last_output_path.open("r", encoding="utf-8") as file:
        last_response = json.load(file)

    print(f"\nLast output: {last_output_path}")
    print(json.dumps(last_response, indent=2, ensure_ascii=False))

[
  {
    "input": "input_example_1.json",
    "output": "output_example_1.json",
    "request_mode": "form-fallback",
    "status_code": 200,
    "run_id": "input-01-chest-pain-red-flags",
    "status": "success"
  },
  {
    "input": "input_example_2.json",
    "output": "output_example_2.json",
    "request_mode": "form-fallback",
    "status_code": 200,
    "run_id": "input-02-high-blood-sugar-symptoms",
    "status": "success"
  },
  {
    "input": "input_example_3.json",
    "output": "output_example_3.json",
    "request_mode": "form-fallback",
    "status_code": 200,
    "run_id": "input-03-warfarin-ibuprofen-risk",
    "status": "success"
  },
  {
    "input": "input_example_4.json",
    "output": "output_example_4.json",
    "request_mode": "form-fallback",
    "status_code": 200,
    "run_id": "input-04-copd-heart-failure-breathlessness",
    "status": "success"
  },
  {
    "input": "input_example_5.json",
    "output": "output_example_5.json",
    "request_mode": "form-fal

In [16]:
# Run one input example only, without looping through all files.
# Change this number to run a different file, e.g. 2 for input_example_2.json.
SINGLE_EXAMPLE_NUMBER = 3

input_path = INPUT_DIR / f"input_example_{SINGLE_EXAMPLE_NUMBER}.json"
output_path = OUTPUT_DIR / f"output_example_{SINGLE_EXAMPLE_NUMBER}.json"

if not input_path.exists():
    raise FileNotFoundError(f"Input file not found: {input_path}")

with input_path.open("r", encoding="utf-8") as file:
    payload = json.load(file)

print(f"Running {input_path.name}...")
print("Payload:", json.dumps(payload, indent=2, ensure_ascii=False))

response_json, request_mode, status_code = call_run_endpoint(payload)

with output_path.open("w", encoding="utf-8") as file:
    json.dump(response_json, file, indent=4, ensure_ascii=False)

print(f"Saved {output_path}")
print("Status:", status_code)
print("Run mode:", request_mode)
print("Run ID:", response_json.get("run_id"))

Running input_example_3.json...
Payload: {
  "user_query": "A clause says 'Company may terminate at any time without cause' but another clause requires 60 days notice. Identify whether it is compliant with UAE labour laws",
  "run_id": "legal-input-03-restructuring-termination-process"
}
Saved /Users/vishvajit.phalke/phantom/source_codes/legal_intelligence/legal_ai_repo/output_examples/output_example_3.json
Status: 200
Run mode: form-fallback
Run ID: legal-input-03-restructuring-termination-process


In [10]:
# Optional: preview the currently selected single input file.
preview_path = INPUT_DIR / f"input_example_{SINGLE_EXAMPLE_NUMBER}.json"

with preview_path.open("r", encoding="utf-8") as file:
    preview_payload = json.load(file)

print(json.dumps(preview_payload, indent=2, ensure_ascii=False))

{'user_query': 'A 35-year-old female has been feeling very tired for several weeks and has increased thirst, frequent urination, occasional blurred vision, and some unintentional weight loss. A recent clinic visit told her that her blood sugar and HbA1c were higher than normal, but she does not remember the exact numbers. Please identify likely clinical concerns, warning signs to watch for, and safe follow-up steps.', 'run_id': 'input-02-high-blood-sugar-symptoms'}
